<a href="https://colab.research.google.com/github/volodymyr-holovan/DTA2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [2]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [3]:
# TODO: виведи баланс класів
clients["upgraded"].value_counts(normalize=True)

,proportion
upgraded,
0,0.515556
1,0.484444


✍️ Випиши списки стовпців (знадобляться далі):

> числові: `age`, `tenure`, `usage`, `support`
> категорійні: `plan`, `region`

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [4]:
from sklearn.model_selection import train_test_split

X = clients.drop("upgraded", axis=1)
y = clients["upgraded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

((720, 6), (180, 6))

### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["age", "tenure", "usage", "support"]
cat_cols = ["plan", "region"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

pipe

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'tenure', 'usage',
                                                   'support']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plan', 'region'])])),
                ('model', LogisticRegression(max_iter=1000))])

### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [7]:
pipe.fit(X_train, y_train)

accuracy = pipe.score(X_test, y_test)
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [10]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = pipe.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Confusion matrix:
[[80 13]
 [15 72]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [15]:
from sklearn.metrics import roc_auc_score

proba = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)

print(f"ROC-AUC: {auc:.3f}")

ROC-AUC: 0.927


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [16]:
feature_names = pipe.named_steps["prep"].get_feature_names_out()
coefs = pipe.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs
})

coef_df = coef_df.reindex(coef_df["coefficient"].abs().sort_values(ascending=False).index)
coef_df

,feature,coefficient
2,num__usage,2.027135
1,num__tenure,1.648386
6,cat__plan_сімейний,1.411953
4,cat__plan_базовий,-1.339728
3,num__support,-0.835947
7,cat__region_захід,0.537637
0,num__age,-0.406875
10,cat__region_схід,-0.340620
8,cat__region_південь,-0.189350
9,cat__region_північ,0.136239


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум ознака num__usage, а знижує ознака cat__plan_базовий.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [17]:
new_client = pd.DataFrame([{
    "age": 30,
    "tenure": 24,
    "usage": 120,
    "support": 0,
    "plan": "сімейний",
    "region": "захід"
}])

prediction = pipe.predict(new_client)[0]
probability = pipe.predict_proba(new_client)[0, 1]

print("Рішення:", "перейде на преміум" if prediction == 1 else "не перейде на преміум")
print(f"Ймовірність переходу: {probability:.3f}")

Рішення: перейде на преміум
Ймовірність переходу: 0.995


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [18]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")

print(f"ROC-AUC CV: {scores.mean():.3f} ± {scores.std():.3f}")
print(scores)

ROC-AUC CV: 0.923 ± 0.018
[0.93437152 0.93140527 0.94005685 0.8907428  0.92070158]


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [19]:
# Бонусні експерименти
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

preprocess_no_scale = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe_no_scale = Pipeline([
    ("prep", preprocess_no_scale),
    ("model", LogisticRegression(max_iter=1000)),
])

print("Без масштабування:", cross_val_score(pipe_no_scale, X, y, cv=5, scoring="roc_auc").mean())
print("З масштабуванням:", cross_val_score(pipe, X, y, cv=5, scoring="roc_auc").mean())

Без масштабування: 0.923455589531725
З масштабуванням: 0.9234556047977966


---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?  
   Логістична регресія прогнозує ймовірність належності об'єкта до класу, а не неперервне числове значення. Тому вона використовується для задач класифікації.

2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?  
   `predict_proba` показує ймовірність кожного класу. Для бізнесу це корисно, бо можна оцінювати рівень ризику або зацікавленості клієнта і вибирати різні дії залежно від ймовірності.

3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?  
   Якщо виконати масштабування до поділу, інформація з тестової вибірки потрапить у навчання. Це створює витік даних і дає занадто оптимістичну оцінку моделі.

4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?  
   Логістична регресія використовує коефіцієнти та регуляризацію, тому різні масштаби ознак можуть впливати на результат. Дерева рішень ділять дані за порогами, тому масштаб ознак для них майже не має значення.

5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?  
   Це означає, що за інших однакових умов клієнти з більшим значенням підтримки мають нижчу ймовірність переходу на преміум. Це лише статистичний зв’язок у моделі, а не обов’язково причина поведінки.

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.